# LEXICON-X II | Notebook de trabajo
By Erasmo Gomez Montoya
Sube y extrae el ZIP; usa CPU para baseline. No uses Ω para ajustar.

In [ ]:
# En Colab: subir y extraer el ZIP de estudiante en /content; cambiar directorio si corresponde.
# Los archivos CSV deben estar junto al notebook.
import pandas as pd, numpy as np, re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from pathlib import Path
D=pd.read_csv('corpus_documentos.csv'); V=pd.read_csv('validation_qa.csv',dtype={'answer':str}).fillna('')
X=D.text.tolist(); vectorizer=TfidfVectorizer(ngram_range=(1,2)); M=vectorizer.fit_transform(X)
def lexical(q,k=5):
    scores=linear_kernel(vectorizer.transform([q]),M).ravel()
    return np.argsort(-scores,kind='stable')[:k]
def evaluate(pred,gold):
    assert pred.qid.is_unique and set(pred.qid)==set(gold.qid), 'IDs incompletos o duplicados'
    z=gold.merge(pred,on='qid',suffixes=('_gold','_pred'))
    exact=(z.answer_gold.astype(str).str.strip().str.lower()==z.answer_pred.astype(str).str.strip().str.lower())
    def hit(r):
        g=set(str(r.evidence_ids).split('|'))-{'','nan'}; p=set(str(r.citations).split('|'))-{'','nan'}
        return bool(g&p) if g else not p
    h=z.apply(hit,axis=1)
    return {'EM':float(exact.mean()),'evidence_hit':float(h.mean()),'joint':float((exact&h).mean()),'by_kind':exact.groupby(z.kind).mean().to_dict()}


In [ ]:
display(D.sample(8,random_state=42)); display(V.head()); print(D.groupby(["language","source"]).size())

## Hipótesis
Escribe hipótesis, variable independiente, métricas y criterio de éxito.

In [ ]:
print(D.iloc[lexical(V.question.iloc[0])][["doc_id","text"]])

## Baseline y alternativa
Implementa predict y run. La salida debe tener qid, answer, citations (IDs separados por |). Usa SIN_EVIDENCIA sin citas cuando no exista respuesta. Compara ambos enfoques con evaluate.

In [ ]:
def predict(question):
    raise NotImplementedError("Implementar respuesta y citas")

## Recursos opcionales: embeddings y generación
Descomenta instalación si es necesaria. No son obligatorios; mide latencia y calidad.

In [ ]:
# Opcional: ejecutar en Colab con Internet y GPU si está disponible.
# %pip install -q sentence-transformers transformers sentencepiece
from sentence_transformers import SentenceTransformer
encoder=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
E=encoder.encode(X,normalize_embeddings=True,show_progress_bar=True)
def dense(q,k=5):
    v=encoder.encode([q],normalize_embeddings=True)[0]
    return np.argsort(-(E@v))[:k]


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
name='google/flan-t5-small'; tok=AutoTokenizer.from_pretrained(name); model=AutoModelForSeq2SeqLM.from_pretrained(name)
device='cuda' if torch.cuda.is_available() else 'cpu'; model.to(device).eval()
def generate(q,idx):
    context='\n'.join(f'{D.iloc[i].doc_id}: {D.iloc[i].text}' for i in idx)
    prompt=f'Responde solo con el número solicitado o SIN_EVIDENCIA. Pregunta: {q} Evidencia: {context}'
    inputs=tok(prompt,return_tensors='pt',truncation=True,max_length=512).to(device)
    with torch.no_grad(): out=model.generate(**inputs,max_new_tokens=20)
    return tok.decode(out[0],skip_special_tokens=True)
print(generate(V.question.iloc[0],dense(V.question.iloc[0],5)))


## Ω
Congela configuración en minuto 120; ejecuta sobre protocolo_omega.csv y exporta omega_predictions.csv. No contiene respuestas.